# 🚗 KNN de regresión — Caso MPG

## 📖 Dónde estamos
Llevas dos modelos sobre estos mismos autos:

| Modelo | R² en prueba | RMSE |
|---|---|---|
| Regresión lineal (RFE) | 0.794 | 3.24 |
| Árbol podado | 0.812 | 3.10 |

El árbol ya mostró que soltar la línea recta ayuda. Hoy vas más lejos con un modelo que **no construye ninguna fórmula**:

> ### ❓ ¿Puede KNN predecir mejor que la regresión lineal y que el árbol? ¿Y a qué costo?

## 🧭 Lo que vas a hacer
1. Partir de los datos **ya preparados** en la sesión 1
2. **Escalar** las variables y comprobar por qué es obligatorio
3. Elegir **k** con validación cruzada
4. Comparar KNN contra la regresión lineal, con la misma partición

## ⏱️ Agenda sugerida (3 horas, sesión **asincrónica**)
| Parte | Dónde | Tiempo |
|---|---|---|
| Fundamento: cómo funciona KNN | `fundamentos.ipynb` | 30 min |
| **Caso MPG — KNN de regresión** | `caso_mpg.ipynb` | 50 min |
| ☕ Descanso | | 10 min |
| **Caso Titanic — KNN de clasificación** | `caso_titanic.ipynb` | 50 min |
| Comparación y cierre | `caso_titanic.ipynb` | 25 min |

---
# 0️⃣ Preparación

Ejecuta estas celdas. Las funciones son las de siempre, más un grupo nuevo para modelos de scikit-learn.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de machine learning

Para los modelos de scikit-learn:

| Función | ¿Para qué sirve? |
|---|---|
| `escalar(X_train, X_test)` | Estandariza ambos conjuntos ajustando sólo con train |
| `evaluar_regresion(nombre, y_real, y_pred)` | R², RMSE y MAE |
| `plot_curva_k(ks, scores, metrica)` | Desempeño de validación cruzada para cada k, y regresa el mejor |
| `plot_curva_complejidad(valores, train, test, parametro, metrica)` | Entrenamiento vs prueba: así se ve el sobreajuste |
| `plot_importancias(modelo, columnas)` | Importancia de variables de un árbol |
| `plot_arbol(modelo, columnas, max_depth)` | Dibuja las reglas del árbol |

In [2]:
# Funciones de machine learning: evaluar y diagnosticar modelos de scikit-learn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import plot_tree


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE y MAE de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | MAE {metricas['MAE']}")
    return metricas


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    from sklearn.preprocessing import StandardScaler

    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_curva_k(k_values, scores, metrica='RMSE', menor_es_mejor=True):
    """Desempeño de validación cruzada para cada valor de k en KNN."""
    k_values, scores = list(k_values), list(scores)
    mejor = k_values[int(np.argmin(scores) if menor_es_mejor else np.argmax(scores))]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=k_values, y=scores, mode='lines+markers',
        line=dict(color='steelblue', width=3), marker=dict(size=7), name=f'{metrica} (CV)'
    ))
    fig.add_vline(x=mejor, line_dash='dash', line_color='red', annotation_text=f'mejor k = {mejor}')
    fig.update_layout(
        title=f'Elección de k — {metrica} de validación cruzada',
        xaxis_title='k (número de vecinos)', yaxis_title=f'{metrica} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()
    return mejor


def plot_curva_complejidad(valores, score_train, score_test, parametro='max_depth', metrica='R²'):
    """Desempeño en entrenamiento vs prueba al aumentar la complejidad: así se ve el sobreajuste."""
    datos = pd.DataFrame({parametro: list(valores) * 2,
                          metrica: list(score_train) + list(score_test),
                          'conjunto': ['Entrenamiento'] * len(score_train) + ['Prueba'] * len(score_test)})

    fig = px.line(
        datos, x=parametro, y=metrica, color='conjunto', markers=True,
        title=f'Sobreajuste: {metrica} según {parametro}', template='plotly_white',
        color_discrete_map={'Entrenamiento': 'steelblue', 'Prueba': 'tomato'}
    )
    fig.update_layout(width=900, height=450)
    fig.show()


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)


def plot_arbol(modelo, columnas, max_depth=3, class_names=None):
    """Dibuja los primeros niveles del árbol para poder leer sus reglas."""
    fig, ax = plt.subplots(figsize=(22, 9))
    plot_tree(
        modelo, max_depth=max_depth, feature_names=list(columnas), class_names=class_names,
        filled=True, rounded=True, fontsize=9, impurity=False, ax=ax
    )
    ax.set_title(f'Árbol de decisión — primeros {max_depth} niveles', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 📥 Los datos, ya preparados

Este caso lo trabajaste en la sesión de regresión lineal, así que no repetimos la exploración ni el preprocesamiento: la celda siguiente trae las dos funciones que hacen lo mismo que hiciste a mano.

| Función | ¿Qué hace? |
|---|---|
| `cargar_mpg()` | Descarga la base SQLite y elimina los nulos de `horsepower` |
| `preparar_mpg(df)` | Separa X/y, hace el split 80/20 (`random_state=42`) y codifica `origin` con los autos americanos como referencia |

Usa exactamente la **misma partición** de aquella sesión, así que las métricas son comparables.

In [3]:
# Datos del caso MPG: carga y preparación (idénticas a las de la sesión 1)
import pandas as pd
import requests
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


def cargar_mpg():
    """Descarga la base SQLite del caso MPG y regresa el DataFrame sin nulos (392 autos)."""
    url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
    with open("mpg.db", "wb") as f:
        f.write(requests.get(url).content)

    query = """
    SELECT
        O.mpg,
        O.cylinders,
        O.displacement,
        O.horsepower,
        O.weight,
        O.acceleration,
        O.model_year,
        ORG.origin,
        N.name
    FROM
        Observation AS O
    JOIN
        Origin AS ORG ON O.origin_id = ORG.origin_id
    JOIN
        Name AS N ON O.name_id = N.name_id
    """
    df = pd.read_sql_query(query, sqlite3.connect("mpg.db"))

    # Sólo 'horsepower' tiene nulos (1.5% de las filas): se eliminan
    return df.dropna(subset=['horsepower'])


def preparar_mpg(df, test_size=0.2, random_state=42):
    """Split 80/20 y One-Hot de 'origin' con los autos americanos como referencia.

    Regresa X_train, X_test, y_train, y_test listos para modelar (todo numérico).
    """
    X = df.drop(columns=['mpg', 'name'])
    y = df['mpg']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    ohe = OneHotEncoder(drop=['usa'], sparse_output=False)
    dummies_train = pd.DataFrame(
        ohe.fit_transform(X_train[['origin']]),
        columns=ohe.get_feature_names_out(), index=X_train.index
    )
    dummies_test = pd.DataFrame(
        ohe.transform(X_test[['origin']]),
        columns=ohe.get_feature_names_out(), index=X_test.index
    )

    X_train_enc = pd.concat([X_train.drop(columns=['origin']), dummies_train], axis=1)
    X_test_enc = pd.concat([X_test.drop(columns=['origin']), dummies_test], axis=1)

    print(f"Train: {X_train_enc.shape} | Test: {X_test_enc.shape}")
    print(f"Variables: {list(X_train_enc.columns)}")
    return X_train_enc, X_test_enc, y_train, y_test

In [4]:
# Esta celda va de regalo: léela con calma y ejecútala
df = cargar_mpg()
X_train, X_test, y_train, y_test = preparar_mpg(df)

df.head()

Train: (313, 8) | Test: (79, 8)
Variables: ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin_europe', 'origin_japan']


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


📋 **Recap de la sesión 1:** 392 autos, sin nulos, `origin` convertido a dummies con los americanos como referencia. La partición es **la misma** (`random_state=42`), así que los resultados son comparables.

| Modelo anterior | R² en prueba | RMSE |
|---|---|---|
| Regresión simple (`weight`) | 0.653 | 4.21 |
| **RFE (`weight` + `model_year`)** | **0.794** | **3.24** |
| **Árbol podado** (sesión 3) | **0.812** | **3.10** |

---
# 1️⃣ Escalar: el paso que no se puede saltar (10 min)

En `fundamentos.ipynb` viste que la distancia la domina la variable de escala más grande. En estos datos, `weight` llega a **5,140 libras** y `model_year` a **82**: sin escalar, el peso decidiría casi solo quién es vecino de quién.

In [5]:
# 1. Escala los datos: escalar(X_train, X_test) regresa los dos conjuntos ya estandarizados
# 2. Comprueba el resultado con .describe().round(2).loc[['mean', 'std', 'min', 'max']]
# ✍️ Tu código aquí
X_train_escalado, X_test_escalado = escalar(X_train, X_test)

X_train_escalado.describe().round(2).loc[['mean', 'std', 'min', 'max']]

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_europe,origin_japan
mean,0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.46,-1.21,-1.53,-1.64,-2.68,-1.71,-0.43,-0.50
max,1.48,2.50,3.28,2.56,3.29,1.60,2.35,1.99


✅ Todas las variables quedaron con **media 0** y **desviación 1**, así que ninguna domina la distancia.

---
# 2️⃣ Elegir k con validación cruzada (20 min)

Probamos muchos valores de `k` y nos quedamos con el que da menor **RMSE** promedio en validación cruzada. Ojo: todo esto ocurre **dentro de train**; el conjunto de prueba no se toca.

In [6]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold

valores_k = range(1, 31)
rmse_cv = []

for k in valores_k:
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_train_escalado, y_train,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='neg_root_mean_squared_error'
    )
    rmse_cv.append(-scores.mean())

mejor_k = plot_curva_k(valores_k, rmse_cv, metrica='RMSE')

✍️ **Lee la curva:**
- ¿Qué valor de k eligió la validación cruzada? ¿Con qué RMSE?
- ¿Por qué crees que el error es **alto** con k = 1, si ese modelo acierta todo en entrenamiento?
- ¿Qué pasa con el error cuando k se vuelve muy grande?

- La validación cruzada eligió **k = 4**, con un RMSE de **3.08**.
- Con k = 1 el error es alto (3.76) porque el modelo copia el valor de **un solo vecino**. En entrenamiento acierta todo porque cada auto es su propio vecino más cercano, pero con autos nuevos copia también el "ruido" de ese único auto. Es como memorizar en lugar de aprender.
- Cuando k se vuelve muy grande, el error vuelve a subir (con k = 30 es 3.76 otra vez), porque promedia muchos autos que ya no se parecen entre sí y el modelo se vuelve demasiado simple.

---
# 3️⃣ Ajustar y evaluar (20 min)

In [7]:
# 1. Crea el modelo con el mejor k: KNeighborsRegressor(n_neighbors=mejor_k)
# 2. Entrénalo con los datos ESCALADOS de entrenamiento
# 3. Predice sobre los datos escalados de prueba  → y_pred_knn
# 4. comparacion = []  y agrega evaluar_regresion(f'KNN (k={mejor_k})', y_test, y_pred_knn)
# ✍️ Tu código aquí
knn = KNeighborsRegressor(n_neighbors=mejor_k)
knn.fit(X_train_escalado, y_train)

y_pred_knn = knn.predict(X_test_escalado)

comparacion = []
comparacion.append(evaluar_regresion(f'KNN (k={mejor_k})', y_test, y_pred_knn))
print("RMSE de validacion cruzada con el mejor k:", round(min(rmse_cv), 3))

KNN (k=4): R² 0.8587 | RMSE 2.6856 | MAE 1.951
RMSE de validacion cruzada con el mejor k: 3.082


In [8]:
# Diagnostica el modelo con plot_actual_vs_predicted() y plot_residuals()
# ✍️ Tu código aquí
plot_actual_vs_predicted(y_test, y_pred_knn, title=f'KNN (k={mejor_k}) - Real vs Predicho')
plot_residuals(y_test, y_pred_knn, title=f'KNN (k={mejor_k}) - Residuales')

✍️ **Compara con los modelos anteriores:**
- ¿Qué R² y qué RMSE obtuviste con KNN? ¿Le ganó a la regresión lineal (0.794) y al árbol (0.812)?
- Si le ganó, ¿qué tiene KNN que la regresión lineal no tiene? Piensa en la **forma** que cada modelo puede tomar.

- Con KNN obtuve un **R² de 0.8587** y un **RMSE de 2.69**. Sí le ganó a la regresión lineal (0.794 y 3.24) y al árbol (0.812 y 3.10).
- KNN no tiene una forma fija: para cada auto busca los 4 más parecidos y promedia su rendimiento, así que puede seguir curvas o cualquier patrón que tengan los datos. La regresión lineal está obligada a ser una **línea recta**, y el árbol hace **escalones**. Como la relación del peso y el motor con el rendimiento no es recta, a KNN le va mejor.

---
# 4️⃣ ¿Y si no hubiéramos escalado? (10 min)

Ya lo dijimos en la teoría. Ahora compruébalo con datos reales: el mismo modelo, el mismo k, sin escalar.

In [9]:
# Entrena el mismo KNN, con el mismo k, pero con X_train y X_test SIN escalar.
# Evalúalo y agrégalo a comparacion. ¿Qué tanto cambia?
# ✍️ Tu código aquí
knn_sin_escalar = KNeighborsRegressor(n_neighbors=mejor_k)
knn_sin_escalar.fit(X_train, y_train)

y_pred_sin_escalar = knn_sin_escalar.predict(X_test)
comparacion.append(evaluar_regresion(f'KNN sin escalar (k={mejor_k})', y_test, y_pred_sin_escalar))

print(pd.DataFrame(comparacion))

# para ver que variables tienen los rangos mas grandes
X_train.describe().round(1).loc[['min', 'max', 'std']]

KNN sin escalar (k=4): R² 0.6261 | RMSE 4.3685 | MAE 3.012
                  modelo      R²    RMSE    MAE
0              KNN (k=4)  0.8587  2.6856  1.951
1  KNN sin escalar (k=4)  0.6261  4.3685  3.012


,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_europe,origin_japan
min,3.0,70.0,46.0,1613.0,8.0,70.0,0.0,0.0
max,8.0,455.0,230.0,5140.0,24.8,82.0,1.0,1.0
std,1.7,103.8,38.3,841.1,2.8,3.6,0.4,0.4


✍️ **¿Qué pasó?**
- ¿Cuánto cayó el R² al no escalar?
- ¿Qué variables crees que dominaron la distancia? (Pista: compara los rangos con `X_train.describe()`)

- El R² cayó de **0.8587 a 0.6261**, o sea como 0.23 puntos. El error subió de 2.69 a 4.37 mpg.
- La distancia la dominó **weight**, que va de 1,613 a 5,140 libras (desviación de 841), y después **displacement** (de 70 a 455). Las demás variables, como acceleration o las dummies de origen, que solo van de 0 a 1, casi no cuentan. Entonces sin escalar KNN prácticamente busca vecinos solo por el peso.

---
# 5️⃣ El efecto de k, con datos reales (15 min)

En la teoría lo viste con datos de juguete. Aquí se ve el sobreajuste con los autos.

In [10]:
# Esta celda va de regalo: léela con calma y ejecútala
ks = [1, 2, 3, 4, 6, 10, 15, 20, 30, 50]
r2_train, r2_test = [], []

for k in ks:
    modelo = KNeighborsRegressor(n_neighbors=k).fit(X_train_escalado, y_train)
    r2_train.append(modelo.score(X_train_escalado, y_train))
    r2_test.append(modelo.score(X_test_escalado, y_test))

plot_curva_complejidad(ks, r2_train, r2_test, parametro='k', metrica='R²')

✍️ **Lee la gráfica:**
- ¿Qué R² de entrenamiento tiene k = 1? ¿Y en prueba? ¿Cómo se llama eso?
- ¿Qué pasa con las dos curvas cuando k es muy grande?
- ¿En qué rango de k dirías que está el buen modelo?

- Con k = 1 el R² de entrenamiento es **1.0** (perfecto) y en prueba es **0.827**. Eso es **sobreajuste**: el modelo se memoriza los datos de entrenamiento.
- Cuando k es muy grande, **las dos curvas bajan** (con k = 50, 0.761 en entrenamiento y 0.663 en prueba). El modelo se vuelve demasiado simple y ya no predice bien ni en entrenamiento: eso es **subajuste**.
- El buen modelo está entre **k = 3 y k = 6**, donde la prueba llega a su máximo (0.857 a 0.859) y la diferencia con entrenamiento es chica.

## 📝 Cierre del caso
✍️ Resume en tres líneas: ¿KNN le ganó a la regresión lineal? ¿Qué ganaste y qué perdiste al cambiar de modelo?

Sí, KNN le ganó a la regresión lineal: R² de 0.859 contra 0.794.
Gané precisión: ahora el error típico es de 2.7 mpg en lugar de 3.2.
Perdí la explicación: KNN no da coeficientes, no puedo decir cuánto baja el rendimiento por cada libra, y además hay que escalar y elegir bien k.

Ahora llévalo a clasificación en `caso_titanic.ipynb`.